# EDA analysis notebook

This notebook stands for EDA of 
[this dataset](https://www.kaggle.com/competitions/rossmann-store-sales/data?select=train.csv) (Rossmann Store Sales competition)

## Some notes about dataset

- sales data for 1,115 Rossmann stores
- Forecasting horizon is about 6 weeks (from competition requirements)
- Reliable sales forecasts enable store managers to create effective staff schedules that increase productivity and motivation
- Some stores in the dataset were temporarily closed for refurbishment.
- Note that all schools are closed on public holidays and weekends

## Prelimitary analysis

In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT_PATH = Path("/content/drive/MyDrive/data/store-sales-forecasting/")
df = pd.read_csv('/content/drive/MyDrive/data/store-sales-forecasting/train.csv')
df = df.set_index('Date')
df.index = pd.to_datetime(df.index)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
rows, columns = df.shape

print(f"Rows: {rows}\nColumns: {columns}")

In [ ]:
print(f"total number of stores is {df["Store"].unique().sum()}")

In [ ]:
df.isna().sum()

In [ ]:
print(f'Duplicated rows: {df.index.duplicated().sum()} ({df.index.duplicated().sum() / rows*100:.2f}%)')

as we have multiple stores at 1 day (each row stands for day and store) we have many dublicates in dataset

## Feature preparation (basic)

Extract total sales, date, day of a week and e.c from the dataset

In [ ]:
df["Total sales"] = df.groupby("Date")["Sales"].sum()

In [ ]:
df[~df.index.duplicated()].head()

In [ ]:
df["Day of month"] = df.index.day
df["Month"] = df.index.month
df["Year"] = df.index.year
df["Week of year"] = df.index.isocalendar().week.astype(int)
df["Day of year"] = df.index.dayofyear
df["Quarter"] = df.index.quarter
df["Is weekend"] = (df.index.dayofweek >= 5).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Autumn

df["Season"] = df["Month"].apply(get_season)

In [ ]:
df.columns

In [ ]:
df.head()

## Distribution analysis

Plot the distributions to meet my future work

In [ ]:
def clear_x_labels(ax):
    """Delete x labels except the last one
    Useful if plots shares same X label
    """
    for a in ax[:-1]:
        a.set_xlabel("")
    return ax

def plot_time_series_plots(nrows,ncolumns, df):
    fig, ax = plt.subplots(nrows,ncolumns, figsize=(20,10))
    fig.subplots_adjust(hspace=0.5)
    
    agg_df = df.groupby("Date")["Sales"].agg("sum")
    
    agg_df.plot(title="Sales over time", ax=ax[0])
    agg_df["2013-01-01" : "2013-12-31"].plot(title="Sales over time (2013-2014)", ax=ax[1])
    agg_df["2013-04-01" : "2013-04-30"].plot(title="Sales over time (1 month, April)", ax=ax[2])
    agg_df["2013-02-01" : "2013-02-8"].plot(title="Sales over time (1 week, 1-8 of February)", ax=ax[3])
    
    ax = clear_x_labels(ax)
    
plot_time_series_plots(4,1,df)
plt.show()

From the plots we see some essential pattern - sales have visible weekly pattern. We see peaks nearly at the beginning of week.

Note that at sundays stores are closed - sales are 0

Sales near 1st of January are higher than usual (may be because of winter celebrations impact)

In [ ]:
store_sales = df.groupby("Store")["Sales"].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

store_sales.plot(kind="hist", bins=75, ax=axes[0], edgecolor="black")
axes[0].set_title("Distribution of Total Sales per Store")
axes[0].set_xlabel("Total Sales")
axes[0].set_ylabel("Number of Stores")

top_n = 20
store_sales.tail(top_n).plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title(f"Top {top_n} Stores by Total Sales")
axes[1].set_xlabel("Total Sales")
axes[1].set_ylabel("Store")

plt.tight_layout()
plt.show()

We se that sales by stores distribution is similar to [log-normal distribution](mathworld.wolfram.com/LogNormalDistribution.html). Also there are ~5-10 stores that have anomaly higher sales (> 1.25 * 10e7). Maybe should be treated as outliers in future

### Closed shops analysis

I have completed some additional research about closed shops. Check notebooks/Closed_shop_analysis for details

Main insights:
- Ration closed / open is about 16%.
- Stores are working Monday-Saturday.
    - But some stores work (~3%) at sundays. These stores are big and have high traffic

In [ ]:
open_df = df[df["Open"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(open_df["Sales"], bins=60, edgecolor="black", alpha=0.7)
axes[0].set_title("Sales Distribution (Open Stores)")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Frequency")

axes[1].hist(open_df["Customers"], bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].set_title("Customers Distribution (Open Stores)")
axes[1].set_xlabel("Customers")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Open stores distribution seems also similar to log-norm distribution. Both customers and sales distributions are without anomaly outliers

In [ ]:
open_df = df[df["Open"] == 1].copy()
season_labels = {0: "Winter", 1: "Spring", 2: "Summer", 3: "Autumn"}
open_df["SeasonLabel"] = open_df["Season"].map(season_labels)

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

sns.boxplot(data=open_df, x="DayOfWeek", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Day of Week")

sns.boxplot(data=open_df, x="Promo", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Promo")
axes[1].set_xticks([0,1])
axes[1].set_xticklabels(labels=["No Promo", "Promo"])

sns.boxplot(data=open_df, x="SchoolHoliday", y="Sales", ax=axes[2])
axes[2].set_title("Sales by School Holiday")
axes[2].set_xticks([0,1])
axes[2].set_xticklabels(["No Holiday", "Holiday"])

sns.boxplot(data=open_df, x="Month", y="Sales", ax=axes[3])
axes[3].set_title("Sales by Month")

sns.boxplot(data=open_df, x="SeasonLabel", y="Sales", ax=axes[4],
            order=["Winter", "Spring", "Summer", "Autumn"])
axes[4].set_title("Sales by Season")

axes[5].set_visible(False)

plt.tight_layout()
plt.show()


We see that high impact are get with:
- promo stores
- Month before new year (November and December)
- At Mondays and Sundays sales deviation slightly different

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.boxplot(data=open_df, x="StateHoliday", y="Sales", ax=axes[0])
axes[0].set_title("Sales by State Holiday")

sns.boxplot(data=open_df, x="Is weekend", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Weekend")
axes[1].set_xticks([0,1])
axes[1].set_xticklabels(["Weekday", "Weekend"])

plt.tight_layout()
plt.show()

#TODO: update x labels with real names

## Stationary analysis

In [ ]:
from statsmodels.tsa.stattools import adfuller

agg_df = df.groupby("Date")["Sales"].agg("sum")
agg_df = agg_df.sort_index()

result = adfuller(agg_df, autolag="AIC")

#FIXME: not true
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"   {key}: {value:.4f}")


Despite having season patterns sales mean stays fixed during long time. The ADF statistics says sales data time series is stationary. No need to remove a trend (no differencing needed)

## Feature correlation analysis

Analysis below is performed on **open stores only** (`Open == 1`) - closed days force
`Sales` to zero and distort correlations.

- `StateHoliday` is label-encoded (`0/a/b/c -> 0/1/2/3`) to join numeric correlations
- Dependence is measured three ways: Pearson (linear), Spearman (monotonic) and mutual information (arbitrary nonlinear)


In [ ]:
import numpy as np
from sklearn.feature_selection import mutual_info_regression

corr_df = open_df.copy().drop(columns=["Total sales"])
corr_df["StateHoliday"] = (
    corr_df["StateHoliday"]
    .astype(str)
    .map({"0": 0, "a": 1, "b": 2, "c": 3})
    .astype(int)
)

features = [
    "Sales",
    "Customers",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
    "DayOfWeek",
    "Day of month",
    "Month",
    "Year",
    "Week of year",
    "Day of year",
    "Quarter",
    "Is weekend",
    "Season",
]
corr_df = corr_df[features]


In [ ]:
spearman_corr = corr_df.corr(method="spearman")

heatmap_kwargs = dict(
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.8},
)
fig, ax = plt.subplots(2,1, figsize=(8, 8))
sns.heatmap(spearman_corr, **heatmap_kwargs, ax=ax[0])
ax[0].set_title("Spearman rank correlation")

target_corr = pd.DataFrame({"Spearman":spearman_corr["Sales"].drop("Sales")})

target_corr = target_corr.loc[target_corr["Spearman"].abs().sort_values(ascending=False).index]

x = np.arange(len(target_corr))

ax[1].bar(x, target_corr["Spearman"])

ax[1].set_xticks(x)
ax[1].set_xticklabels(target_corr.index, rotation=45, ha="right")
ax[1].axhline(0, color="black", linewidth=0.8)
ax[1].set_ylabel("Correlation with Sales")
ax[1].set_title("Feature-to-target correlation ranking (Spearman)")

plt.tight_layout()
plt.show()


Looking at correlation matrix we see that we have many multicollinearities (Promo and weekends and time features). The spearman corelation catch data relation direction. This approach much more flexible compare to catch linear correlation.

We see that highest target correlated feature is customers amount. I can use lags sales and customers both to forecast sales. The highest categorical feature is `Promo` availability

In [ ]:
fig = plt.figure(figsize=(17, 7))
gs = fig.add_gridspec(
    2, 4,
    width_ratios=(4, 1, 4, 1),
    height_ratios=(1, 4),
    left=0.06, right=0.96, bottom=0.08, top=0.90,
    wspace=0.05, hspace=0.05,
)

promo_conditions = [(0, "No Promo"), (1, "Promo")]

for main_col, (promo_val, label) in zip([0, 2], promo_conditions):
    sub = open_df[open_df["Promo"] == promo_val]

    ax_main = fig.add_subplot(gs[1, main_col])
    ax_top = fig.add_subplot(gs[0, main_col], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1, main_col + 1], sharey=ax_main)

    hb = ax_main.hexbin(
        sub["Customers"], sub["Sales"],
        gridsize=70, mincnt=1, bins="log", cmap="viridis",
    )
    ax_top.hist(sub["Customers"], bins=80, color="steelblue")
    ax_right.hist(sub["Sales"], bins=80, orientation="horizontal", color="steelblue")

    ax_top.axis("off")
    ax_right.axis("off")

    ax_main.set_title(f"Customers vs Sales ({label})", fontsize=12)
    ax_main.set_xlabel("Customers")
    ax_main.set_ylabel("Sales")
    fig.colorbar(hb, ax=ax_right, label="count (log scale)")

fig.suptitle("Customers vs Sales density (open stores)", fontsize=14)
plt.show()


This plot also proves `Promo` label importance. We see that at `Promo` stores people tends to buy more products (Compare 5000 sales / 1000 customers distribution on the left side with 10000 sales / 1000 customers). Approximately people sales at promo are 1.5-2 times higher than without it 

In [ ]:
X = corr_df.drop(columns=["Sales"])
y = corr_df["Sales"]

mi_path = ROOT_PATH / "mi_scores.csv"

if mi_path.exists():
   mi_scores = pd.read_csv(mi_path, index_col=0).squeeze()
else:
   mi_scores = pd.Series(
      mutual_info_regression(X, y, random_state=42),
      index=X.columns,
      name="Mutual information",
   ).sort_values(ascending=False)
   mi_scores.to_csv(mi_path)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=mi_scores.values, y=mi_scores.index, ax=ax, color="steelblue")
ax.set_title("Mutual information with Sales")
ax.set_xlabel("MI score")

plt.tight_layout()
plt.show()


The mutual information gives more details about `Sales`. As mutual information catch *all* kinds of relations it gives us the deeper pattern analysis. Mutual information can catch U-like, periodic and seasonal patterns and extremely useful for us right now.

Notice that we see an day of year there. Mutual information gave `Day of year` feature a high weight (more that `Promo` one). It make sense: day of year patterns (New Year and e.t.c) are important but have noisy not linear structure that Spearman do not get. Also mutual information gives date features bigger weight (But hierarchy stays mostly same)

In [ ]:
daily_sales = df.groupby("Date")["Sales"].sum()

fig, ax = plt.subplots(2,1, figsize=(10,5))

pd.plotting.autocorrelation_plot(daily_sales, ax=ax[0])
ax[0].set_title("Autocorrelation of daily total sales (Unzoomed)")
ax[0].set_xlabel("Lag (days)")
ax[0].set_ylabel("Autocorrelation")

pd.plotting.autocorrelation_plot(daily_sales, ax=ax[1])
ax[1].set_xlim(0, 100)
ax[1].set_title("Autocorrelation of daily total sales (Zoomed)")
ax[1].set_xlabel("Lag (days)")
ax[1].set_ylabel("Autocorrelation")

plt.tight_layout()
plt.show()

**Autocorrelation.** Correlate daily total sales with its own past values. A weekly
periodicity shows up as a strong peak at lag 7 (and multiples). But the real reason of this peak is open / closed stores at sundays. From open / closed shops analysis notebook we know that only ~15 stores (from 1100) are opened at sundays.


However wee see that predicting with 6 week horizon will be difficult task. The amplitude at 42 day is ~-0.1-0.1 means that autocorrellation is not statistically distinguishable from noise at this horizon

In [ ]:
sales_not_6 = daily_sales[daily_sales.index.day_of_week != 6]

fig, ax = plt.subplots(2,1, figsize=(10,5))

pd.plotting.autocorrelation_plot(sales_not_6, ax=ax[0])
ax[0].set_title("Autocorrelation of daily total sales without sunday (Unzoomed)")
ax[0].set_xlabel("Lag (days)")
ax[0].set_ylabel("Autocorrelation")

pd.plotting.autocorrelation_plot(sales_not_6, ax=ax[1])
ax[1].set_xlim(0, 100)
ax[1].set_title("Autocorrelation of daily total sales without sunday (Zoomed)")
ax[1].set_xlabel("Lag (days)")
ax[1].set_ylabel("Autocorrelation")

plt.tight_layout()
plt.show()

We see that without sundays amplitude come down from 0.7 to 0.2. That means sundays closings were committing impact to lags.

Also note as I removed sundays from dataset. The lags at plots cannot be interpretated fully. Calendar grid had been restructured.

### Seasonal decomposition

Decompose daily total sales into **trend**, **weekly seasonal**, and **residual** components.

Two methods are compared:
- **Classical additive** (`seasonal_decompose`) — simple moving-average based; fast but sensitive to outliers.
- **STL** (Seasonal-Trend decomposition using Loess) — robust to outliers, handles evolving seasonality.

Each method is shown on a **single year** (first 365 days) to inspect the weekly pattern up close,
and on the **full series** to see the long-run trend.

In [ ]:
%pip install -q statsmodels

from statsmodels.tsa.seasonal import seasonal_decompose, STL

def plot_decomposition(series, res, series_name, titles=(
        "Observed", "Trend", "Seasonal (weekly)", "Residual"), figsize=(16, 12)):
    """Side-by-side stacked plot of a time series and its decomposition parts."""
    fig, axes = plt.subplots(len(titles), 1, figsize=figsize, sharex=True)
    parts = (series, res.trend, res.seasonal, res.resid)
    for ax, (name, part) in zip(axes, zip(titles, parts)):
        part.plot(ax=ax, linewidth=0.8)
        ax.set_title(name)
        ax.set_ylabel("Sales")
    axes[-1].set_xlabel("Date")
    fig.suptitle(series_name, y=0.995)
    plt.tight_layout()
    plt.show()

res = seasonal_decompose(daily_sales.iloc[:365], model="additive", period=7,
                         extrapolate_trend="freq")
plot_decomposition(daily_sales.iloc[:365], res,
                   "Classical additive decomposition (1 year)")

In [ ]:
res = seasonal_decompose(daily_sales, model="additive", period=7,
                     extrapolate_trend="freq")
plot_decomposition(daily_sales, res,
                   "Classical additive decomposition (full series)",
                   titles=("Trend", "Residual"), figsize=(16, 12))

In [ ]:
res = STL(daily_sales[:365], period=7, robust=True).fit()
plot_decomposition(daily_sales[:365], res,
                   "STL decomposition (1 year)")

In [ ]:
res = STL(daily_sales, period=90, robust=True).fit()
plot_decomposition(daily_sales, res, "STL decomposition (full series)")

Увеличь сглаживание тренда (trend / trend_deg):
В statsmodels.tsa.seasonal.STL увеличь параметр trend (сделай его больше, это должно быть нечетное число, существенно превышающее период сезонности). Это насильно сгладит линию.

Сделай сезонность более гибкой (seasonal):
Если амплитуда недельных пиков меняется от месяца к месяцу, увеличивай окно seasonal (например, seasonal=13 или выше), чтобы сгладить изменения компонент.

Учти годовую сезонность (MSTL):
Для дневных данных с недельной и годовой цикличностью стандартного STL часто не хватает. Используй MSTL (from statsmodels.tsa.seasonal import MSTL), передав список периодов: periods=(7, 365).

Удали крайние выбросы или примени Robust STL:
Передай robust=True в STL, чтобы разовые праздничные пики (как в январе 2014) не ломали всю декомпозицию.

The **STL** decomposition gives a cleaner, more stable trend than the classical
method because it is robust to the outliers introduced by Sunday closures and
holiday spikes. Both methods confirm a clear **weekly seasonal component** (Sunday dips)
and a **flat long-run trend** - matching the stationarity result from the ADF test above.
The residual stays bounded, so the additive model captures the repeatable signal well -
the same pattern that drives the lag features in `Baseline.ipynb`.

### Takeaways

- `Customers` dominates - the strongest linear, monotonic and nonlinear association with `Sales`
- `Promo` is the strongest categorical driver (positive); `StateHoliday` acts negatively
- Date features (`Month`, `Quarter`, `Week of year`, `Day of year`) are individually weak but heavily collinear with each other - keep only a subset for modeling or rely on tree models
- Large Pearson vs Spearman gaps signal monotonic-but-nonlinear relationships worth a transform (e.g. log of `Sales` / `Customers`)
- Raw Pearson overstates the count-like features: once `Sales`/`Customers` are `log1p`-transformed, the coefficients drop, confirming that part of the raw value was tail-driven skew, not real linearity.
- The flat heatmap can't see time structure. Daily total sales show a strong weekly autocorrelation peak at lag 7 (and multiples) and a clean weekly seasonal component - the repeatable signal that matters for the ~6-week horizon and that motivates the lag features in `Baseline.ipynb`.
